# EXECUTE SPARK IN MY LOCAL HOST

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

# IMPORT ALL LIBRARIES YOU NEED

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import  *
from pyspark.sql.types import StructField, StructType, IntegerType, \
    StringType, DoubleType
from pyspark.sql.dataframe import DataFrame
from pyspark.sql import functions as F

# START A SPARK SESSION

In [3]:
def spark_session(app_name: str = "ATMWithDrawlMonitoring") -> SparkSession:
    """
    Create and return a SparkSession with the specified application name.
    """
    
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName(app_name)
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.driver.memory", "1g")
        .getOrCreate()
    )
    return spark


# LOAD CSV DOCUMENT AND CONVERT IT TO DATAFRAME

In [4]:
def read_csv(spark: SparkSession, file_path: str) -> DataFrame:
    """
    file_path: read document local path
    header=True: confirm the document has headers
    inferSchema=True: read type headers from csv document (NOT RECOMMENDED, BECAUSE IF THERE ARE MILLIONS OF RECORDS, IT IS VERY SLOW)
    schema=schema: read type headers from csv document (BEST PRACTICE TO MILLIOS OF RECORDS)
    sep=",": SEPARATOR OF DATA FROM MY CSV DATA TO DATAFRAME
    """
    # NOTE: IT'S BETTER TO CREATE A DATA SCHEMA TO MY CSV DOCUMENT DATAFRAME
    
    # CREATE A DATA SCHEMA TO MY DATAFRAME FROM THE CSV DOCUMENT (BEST PRACTICE)
    schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("date", StringType(), True),
        StructField("product", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("quantity", IntegerType(), True)
    ])
    
    df = spark.read.csv(file_path, header=True, schema=schema, sep=",")
    
    return df

# APPLY DATE TRANSFORMATIONS

In [5]:
def date_transform(df: DataFrame) -> DataFrame:
    df_cleaned = df.withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    
    return df_cleaned

# APPLY FILTER DATA GREATER THAN 100 

In [6]:
def filter_data(df: DataFrame) -> DataFrame:
    df_filter = df.withColumn("total", F.col("price") * F.col("quantity")) \
                .filter(F.col("total") > 100)
    
    return df_filter

# MAIN FUNCTION TO EXECUTE SPARK DATA CLEANSY AND CONVERT

In [7]:
def main():
    spark = spark_session()
    try:
        print("------------ READ CSV TO DATAFRAME ---------------")
        df = read_csv(spark,file_path="C:/temp/sales.csv")
        df.show(5, truncate=False)
        
        print("------------- DATE TRANSFORM ---------------")
        basic_transform = date_transform(df)
        basic_transform.select("id", "product", "price", "quantity").show(5, truncate=False)
        
        print("------------ FILTER TOTAL GREATER THAN 100 ---------------")
        df_filter = filter_data(df)
        df_filter.select("id", "date", "product", "total").show(5, truncate=False)
    except Exception as e:
        print(f"Error occurred while reading document: {e}")
        spark.stop()

if __name__ == "__main__":
    main()

------------ READ CSV TO DATAFRAME ---------------
+---+----------+--------+------+--------+
|id |date      |product |price |quantity|
+---+----------+--------+------+--------+
|1  |2024-01-10|Portátil|1200.0|1       |
|2  |2024-01-11|Ratón   |25.0  |3       |
|3  |2024-01-12|Teclado |45.0  |2       |
|4  |2024-01-13|Monitor |300.0 |1       |
|5  |2024-01-14|Portátil|1250.0|1       |
+---+----------+--------+------+--------+
only showing top 5 rows
------------- DATE TRANSFORM ---------------
+---+--------+------+--------+
|id |product |price |quantity|
+---+--------+------+--------+
|1  |Portátil|1200.0|1       |
|2  |Ratón   |25.0  |3       |
|3  |Teclado |45.0  |2       |
|4  |Monitor |300.0 |1       |
|5  |Portátil|1250.0|1       |
+---+--------+------+--------+
only showing top 5 rows
------------ FILTER TOTAL GREATER THAN 100 ---------------
+---+----------+-----------+------+
|id |date      |product    |total |
+---+----------+-----------+------+
|1  |2024-01-10|Portátil   |1200